# Conversational AI (26/27) Assignment 4: Prompting
This notebook uses Qwen to generate a dialogue utterance by providing various instructions. 
The dataset used in the assignment is the one from [DSTC11's task 5](https://github.com/lkra/dstc11-track5/blob/main/data), same as assignments 1-3.


## Assignment
In this assignment, we will load a large language model (Qwen) and ask it to generate a dialogue response using different instructions and examples. 
The notebook compares four different prompting techniques:
* Zero-shot prompting
* Few-shot prompting
* Chain-of-Thought prompting
* Chain-of-Thought with Self-Consistency
* Tree-of-Thoughts prompting

For each of these techniques, we explore how to interact with the model, design prompts, and interpret the response.

**NOTE:** In addition to the most common approaches, there are other, less widely adopted prompting techniques, such as *CARP* (Chain-of-Thought with Retrieval Augmented Prompting) or *LPP* (Least-to-Most Prompting). It is important to remember that prompting is a flexible strategy that can be tailored to the specific needs of classification tasks.

This notebook covers the following aspects:

1. Implementation of popular prompting methods (from lecture 4)
2. Evaluation with multiple metrics (from assignment 3)
3. Analyzing the model behavior across configurations

**Assignment steps:**

First, review this notebook to ensure you understand its purpose.

Throughout the notebook, you will find questions you need to answer to complete the assignment. These are both coding questions and questions that evaluate your understanding of the data, the process, and the model behavior. These questions are indicated as <span style="background:yellow">__Q#__</span>. This notebook contains 11 such questions and tasks:

1. Complete the prompting function and run zero- and few-shot prompting with Qwen2.5-3B and Qwen3-1.7B. **(Q1, Q2)**
2. Compare zero-and few-shot performance of Qwen2.5-3B and Qwen3-1.7B. **(Q3)**
3. Complete the chain-of-thought code. **(Q4)**
4. Complete the chain-of-thought with Self-Consistency code. **(Q5)**
5. Answer the questions about the comparison of the methods. **(Q6 - Q11)**
6. Review the Tree-of-Thoughts code, run it, and compare it to chain-of-thought prompting. **(Q12, Q13)**

## Reminders about Kaggle notebooks
1. Pay attention to usage statistics, especially memory, CPU, and GPU
2. Pay attention to the quota of GPU (measured in hours)
3. "Turn OFF" the internet after each session. Turn on the internet when starting a session.
4. "Turn off" the accelerator after each use. Turn the accelerator on when starting a session.


## Ready?
Let's get started! There are four preparations we will start with:
1. installing packages
2. importing relevant packages
3. setting up the directory with the task data
4. defining auxiliary LLM functions

## Installation

The Kaggle notebooks use a Python 3 environment, and they are already "pre-loaded" with various analytic Python packages, like Json, Pandas, and Numpy. If you are curious, you can see the package definition in [this repository](https://github.com/kaggle/docker-python).

We will install a few specialized packages for evaluation and for working with language models.

Let's just double check that Python is set up and we are using a relatively new version (like 3.10):

In [ ]:
!python --version

In [ ]:
# To use BLEU and ROUGE
!pip install sacrebleu
!pip install rouge

# To use BERTSCORE
!pip install evaluate
!pip install bert_score

# To use ْQwen
!pip install transformers

# To calculate cosine similarities between sentences
!pip install sentence-transformers

## Imports

The following code loads several standard packages, packages for evaluation, and packages for working with LLMs.

In [ ]:
import numpy as np 
import json
import os
import shutil
import subprocess

from sacrebleu.metrics import BLEU
from rouge import Rouge

from evaluate import load

from sentence_transformers import CrossEncoder

from transformers import AutoModelForCausalLM, AutoTokenizer

bleu_scorer = BLEU()
rouge_scorer = Rouge()
bertscore = load("bertscore")


## Clone the data/task repository

In [ ]:
def setup_repo(repo_url: str, repo_name: str, work_dir: str = "/kaggle/working"):
    os.chdir(work_dir)
    
    # Remove repo if it exists
    if os.path.exists(os.path.join(work_dir, repo_name)):
        shutil.rmtree(os.path.join(work_dir, repo_name))
    
    # Clone repo
    subprocess.run(["git", "clone", repo_url], check=True)
    
    # Move into repo/data
    os.chdir(os.path.join(repo_name, "data"))


setup_repo("https://github.com/lkra/dstc11-track5.git", "dstc11-track5")

In [ ]:
## List all files in the current directory iteratively:
for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

There are two files that are vital to this notebook. `Logs` contains the conversation logs, whereas `labels` are the ground-truth responses by humans. Feel free to inspect the contents of these files at this point.

In [ ]:
with open('train/logs.json', 'r') as f:
    contexts=json.load(f)

with open('train/labels.json', 'r') as f:
    labels=json.load(f)

## Helper functions

In this section, we define several utility functions that will be used later in the assignment. You may need to get back to this section for some of the assignment steps.

### Evaluation functions

We will start by defining wrapper functions for our evaluation metrics. ROUGE, BLEU, and Bertscore are implemented fully in this notebook already. We suggest you to spend a few minutes to call these functions with some sample inputs to understand how they work.

In [ ]:
def compute_rouge(hypothesis, reference):
    """
    Returns the prediction and the reference for a single dialogue, returns their ROUGE score.
    """
    score = rouge_scorer.get_scores(
          hyps=hypothesis,
          refs=reference,
      )
    return score[0]["rouge-l"]["f"]

def compute_bleu(hypothesis, reference):
    """
    Returns the prediction and the reference for a single dialogue, returns their BLEU score.
    """
    score = bleu_scorer.sentence_score(
        hypothesis=hypothesis,
        references=[reference],
      )
    return score.score/100 # sacreBLEU gives the score in percent

def compute_bertscore(predictions, references):
    """
    Receives two lists of strings with equal length. Returns their pairwise Bertscores (precision, recall, F1 score).
    """
    results = bertscore.compute(predictions=predictions, references=references, lang="en")
    return results

### LLM functions


 * **get_qwen3_model** and **get_qwen2_model:** Load the Qwen models and assign them to global variables.
 * **query_qwen:** Get the output of the models for a given prompt.


In [ ]:
_qwen3_model = None
_qwen3_tokenizer = None

_qwen2_model = None
_qwen2_tokenizer = None


def get_qwen3_model():
    global _qwen3_model, _qwen3_tokenizer
    if _qwen3_model is None or _qwen3_tokenizer is None:
        model_name = "Qwen/Qwen3-1.7B"
        _qwen3_tokenizer = AutoTokenizer.from_pretrained(model_name)
        _qwen3_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
    return _qwen3_model, _qwen3_tokenizer


def get_qwen2_model():
    global _qwen2_model, _qwen2_tokenizer
    if _qwen2_model is None or _qwen2_tokenizer is None:
        model_name = "Qwen/Qwen2.5-3B-Instruct"
        _qwen2_tokenizer = AutoTokenizer.from_pretrained(model_name)
        _qwen2_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
    return _qwen2_model, _qwen2_tokenizer

For Qwen3-1.7B, we set the *enable_thinking* parameter to *false* because enabling it makes the model take much longer to produce an answer, and we don’t need that level of detailed reasoning for this task

In [ ]:
def query_qwen(messages, model_name):
    
    if model_name == "Qwen3-1.7B":
        model, tokenizer = get_qwen3_model()

        text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
        )
        
    elif model_name == "Qwen2.5-3B":
        model, tokenizer = get_qwen2_model()

        text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
        )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=2000
    )

    output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]
    output = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    return output

Next, we define functions to prepare the examples.

 * **get_examples:** retrieves a subset of context–label pairs with valid targets from the dataset.

 * **prepare_examples:** formats the retrieved pairs into structured textual demonstrations suitable for few-shot prompting.


In [ ]:
def prepare_examples(chat, gt_label):
    """
    Transforms a chat into a demonstration example for an LLM. 
    Returns a demonstration sample for a single prior chat.
    """
    demo=""
    for ut in chat:
        role = "user" if ut['speaker']=="U" else "system"
        demo+= f'{role}: {ut["text"]}\n'
    demo+=f'Response: {gt_label}\n'
    return demo


def get_examples(contexts, labels, n_examples):
    """
    Get `n_examples` chats with labels, ignoring those where we have no response. 
    To make sure that we don't pollute the evaluation, 
    we obtain examples from the end of the dataset. 
    Returns a list of example chats and a list of example labels.
    """
    example_chats=[]
    example_labels=[]

    for c, l in zip(reversed(contexts), reversed(labels)):
        if l['target']:
            example_chats.append(c)
            example_labels.append(l)
            if len(example_chats)==n_examples:
                break
    return example_chats, example_labels
    


<span style="background:yellow">__Q1:__ Complete the `prepare_messages` function, which is necessary for the prompting to work.</span>

**prepare_messages:** this function assembles the complete prompt for the model, combining the initial instruction, training examples, and the history of the current conversation. You need to define the initial instruction and training examples, and ensure that each turn in the conversation history is properly mapped to a message with the correct role (user or assistant) and its corresponding content.

In [ ]:
def prepare_messages(history, example_chats=[], example_labels=[]):
    """
    Prepare the messages that the model will get as input. 
    Considers the primary model instruction, demonstrations for few-shot settings, 
    and the history of the current chat. Returns the list of messages.
    """
    
    
    #TODO: write a suitable instruction for the model.
    instruction=f'##Here##'
    
    if len(example_chats): # for few-shot only
        #TODO: Complete this function for few-shot
        instruction+=f'##Here##'
        example_counter = 1
        for chat, label in zip(example_chats,example_labels):
            formatted_example=prepare_examples(##Here##)
            instruction+=f'Example {example_counter}:\n{formatted_example}\n'
            example_counter += 1
    
    # Construct dialogue history
    messages=[]
    messages.append({"role": "system", "content": instruction})
    for ut in history:
        ##TODO: For each dialog history, you should add a message, indicating the "role" (which could be user or assistant) and the "content"
        role = ##here##
        messages.append({"role": role, "content": ##here##})

    return messages

## Zero- and few-shot model run and evaluation

<span style="background:yellow">__Q2:__ Use the helper functions to run Qwen2.5-3B and Qwen3-1.7B  in this block. Make sure your code shows both their zero- and few-shot evaluations (adapt the code and add functions where necessary).</span>

* Construct the prompt (including demonstrations when applicable).

* Execute each model on `n_dialogues` = 100.

* Evaluate the model outputs and report metrics separately for each model and regime (zero-shot vs. few-shot).

* **evaluate_answers:** this function evaluates how similar a model’s output (model_response) is to a ground truth answer (gt_response) using the metrics ROUGE and BLEU.

In [ ]:
def evaluate_answers(model_response, gt_response):
    rouge=compute_rouge(model_response, gt_response)
    bleu=compute_bleu(model_response, gt_response)

    print("MODEL:", model_response)
    print("GROUND TRUTH:", gt_response)
    print("ROUGE:", rouge, "BLEU:", bleu)

    print()

    return rouge, bleu
    

* **final_evaluation:** performs an evaluation of the model’s performance, reporting the average BLEU, ROUGE, and BERTScore.

In [ ]:
def final_evaluation(all_bleus, all_rouges, all_predictions, all_references):
    print("Mean BLEU", np.mean(all_bleus))
    print("Mean ROUGE", np.mean(all_rouges))
    print("BERTscore", np.mean(compute_bertscore(all_predictions, all_references)['f1']))

* **run_pipeline:** this function iterates over the dialogs until reaching *n_dialogues*. It builds the prompt (zero-shot or few-shot) and generates the model's response (*query_qwen*). Then, it evaluates that response with ROUGE and BLEU metrics. Finally, it saves predictions, references, and metrics in lists and prints the progress and results of each dialog.

In [ ]:
def run_pipeline(n_dialogues, n_examples, model_name):
    if n_examples>0:
        example_chats, example_labels=get_examples(contexts, labels, n_examples)
    
    all_bleus=[]
    all_rouges=[]
    all_predictions=[]
    all_references=[]
    
    counter_dialogues = 0
    idx_dialogue = 0

    while counter_dialogues < n_dialogues:
        ## Get a valid dialogue
        chat = contexts[idx_dialogue]
        if 'response' not in labels[idx_dialogue]:
            idx_dialogue += 1
            continue
        gt_response = labels[idx_dialogue]['response']
        counter_dialogues += 1
        idx_dialogue += 1
        
        print("Processing dialogue", counter_dialogues)
    
        ## prepare messages
        messages = prepare_messages(chat, example_chats, example_labels) if n_examples > 0 else prepare_messages(chat)
        
        ## get the model's output
        model_response = query_qwen(messages, model_name)
            
        ## Evaluation
        rouge, bleu = evaluate_answers(model_response, gt_response)
    
        ## save the results
        all_predictions.append(model_response)
        all_references.append(gt_response)
        all_bleus.append(bleu)
        all_rouges.append(rouge)

    final_evaluation(all_bleus, all_rouges, all_predictions, all_references)
    

Let's print all our metrics for this configuration:

In [ ]:
n_dialogues = 100
n_examples = 3 # 0 for zero-shot; >0 for few-shot
model_name = "Qwen3-1.7B" ## "Qwen2.5-3B"

run_pipeline(n_dialogues, n_examples, model_name)

<span style="background:yellow">__Q3:__ List the results you obtain for both models for zero-shot prompting on the automated metrics. Then, briefly reflect: how high are the scores, which model does better, and why may that be?</span>

```
# Your answer here
```

## Chain-of-Thought
<span style="background:yellow">__Q4:__ Incorporate chain-of-thought prompting in the instruction field. Use the provided instruction text below. Adapt the response parsing code to evaluate only the generation within `<output>` tags.</span>

What you need to do:

- Create a new function, similar to prepare_messages, that uses the new prompt.
- Create another function, similar to run_pipeline, that calls this new prepare_messages function.
- Run the pipeline with both models in a zero-shot setting.
- Extract the output from the <output> tag.
- Check whether the models follow the requested format and examine how their outputs differ.
- Get new results

In [ ]:
# from https://ollama.com/zcohn/llama3.1-uncensored-cot/blobs/54b94c1841a9
chain_of_thought_instruction="""
1. Begin with a <thinking> section. 
2. Inside the thinking section: 
    a. Briefly analyze the question and outline your approach. 
    b. Present a clear plan of steps to solve the problem. 
    c. Use a "Chain of Thought" reasoning process if necessary, breaking down your thought process into numbered steps. 
3. Include a <reflection> section for each idea where you: 
    a. Review your reasoning. 
    b. Check for potential errors or oversights. 
    c. Confirm or adjust your conclusion if necessary. 
4. Be sure to close all reflection sections. 
5. Close the thinking section with </thinking>. 
6. Provide your final answer in an <output></output> section.

Remember:
Always use these tags in your responses. 
Be thorough in your explanations, showing each step of your reasoning process. 
Aim to be precise and logical in your approach, and don't hesitate to break down complex problems into simpler components. 
Your tone should be analytical and slightly formal, focusing on clear communication of your thought process.

Remember: 
<thinking> and <reflection> and <output> MUST be tags and must be closed at their conclusion. 
Make sure all <tags> are on separate lines with no other text. Do not include other text on a line containing a tag.
You MUST provide your final answer inside the <output></output> tags. 

Provide your output in the following format:
<thinking>
[your initial thinking goes here]
</thinking>

<reflection>
[Review your previous reasoning, either confirm or adjust]
</reflection>

<output>
[Provide your final answer here.]
</output>

As shown in the example above, all three tags must be there, and inside them, you must fill in the requested information.
DO not provide any information outside the tags. All the information must go in the suitable tag.

"""

# Your solution here.


## Chain-of-Thought (CoT) with Self-Consistency
<span style="background:yellow">Q5: Complete the code to implement Chain-of-Thought with Self-Consistency by following the instructions below.</span>

**Chain-of-Thought (CoT) with Self-Consistency** is a prompting and decoding technique that generates **multiple reasoning paths** for the same problem and selects the most consistent final answer, typically through majority voting, rather than relying on a single model response.

Your task:

- Set the temperature: Choose a temperature greater than 0 (preferably above 0.6) to encourage diverse responses.
- Generate multiple responses: Use your CoT prompt to call the model **n** times, where n = 3 or 5.
- Collect the responses: Store all n generated responses in a list.

In [ ]:
temperature = # Your value
n = # Your value (3 or 5)
n_samples = 10

def query_qwen_SC(messages, temperature, model_name = "Qwen3-1.7B"):
    
    if model_name == "Qwen3-1.7B":
        model, tokenizer = get_qwen3_model()

        text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
        )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2000,
    do_sample=True,
    temperature=temperature
    )

    output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]
    output = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    return output



# TODO:
# 1. Select 10 valid dialogues from the dataset.
# 2. For each dialogue, use the CoT prompt from Q4.
# 3. Call the model n times with the same prompt and temperature.
# 4. Store all n generated responses for each dialogue in a list.
# 5. Print the generated responses for comparison.

# Your code here

    

## Questions

**Pick the first 100 samples for automated metric analysis and use them to answer the questions below. When asked for manual evaluation, do this on 5 dialogues.**

Please answer each question with a single paragraph, consisting of 2-5 sentences. Include examples to support your answers. Remember to give examples and include results when describing your findings!



<span style="background:yellow">__Q6:__ Test Qwen3-1.7B and Qwen2.5-3B in few-shot setting with 1, 3, and 5 examples. What do you observe in terms of the evaluation metrics BLEU, ROUGE, [Bertscore](https://huggingface.co/spaces/evaluate-metric/bertscore)? For the remaining questions and experiments, proceed with the language model (Qwen3-1.7B or Qwen2.5-3B) that yielded better scores on this evaluation.</span>

```
# Your answer here
```

<span style="background:yellow">__Q7:__ Now define three quality criteria that you will apply to evaluate manually, by consulting papers by [Deriu et al.](https://link.springer.com/article/10.1007/s10462-020-09866-x) and by [Howcroft et al.](https://aclanthology.org/2020.inlg-1.23/). If you did this in assignment 3, you can just reuse the metrics here. Analyze the generation with 0, 1, and 5 demonstrations using these three manual metrics, for 5 dialogues. Which of the methods seems to be of the highest quality according to your manual assessment?</span>

```
# Your answer here
```

<span style="background:yellow">__Q8:__  Now compare your manual evaluation from Q6 to the results you obtained with the automatic metrics. What do you observe?</span>


```
# Your answer here
```

<span style="background:yellow">__Q9:__ Compare zero-shot chain-of-thought performance with that of standard zero-shot (using both the automatic metrics and manual judgment). What do you observe?</span>


```
# Your answer here
```


<span style="background:yellow">__Q10:__ Repeat the zero-shot experiments using two different temperature parameters: 0 and 1.
For each temperature, run the experiment on 5 dialogues, and repeat each experiment 3 times. Print both the generated outputs and the automatic evaluation metrics. </span>

Afterward, analyze the results:

- What do you observe for each temperature across different runs?
- Do you see consistency in the model’s answers or not?
- How do the metrics behave across runs?

```
# Your answer here
```

<span style="background:yellow">__Q11__: Analyze the results of Chain-of-Thought (CoT) with Self-Consistency for the 10 samples used in Q5.</span>

Examine the generated responses for each sample and answer the following questions:

- How different are the generated responses for the same sample? Do they produce similar answers or significantly different ones?
- Is there a clear majority among the generated answers? If so, can you use the most frequent answer as the final answer through majority voting?

```
# Your answer here
```

## Tree-of-Thoughts

Chain-of-thought prompting asks the model to produce *one* reasoning path in a single pass. If an early step in that path goes wrong (e.g., the model misreads which aspect the user is asking about), the final answer is wrong as well, and the model has no way to go back. **Tree-of-Thoughts (ToT)** [(Yao et al., 2023)](https://arxiv.org/abs/2305.10601), which we discussed in lecture 4, turns reasoning into a search problem instead: the LLM generates several candidate *thoughts* for every step, evaluates how promising each partial reasoning path (a *state*) is, and a search algorithm decides which paths to explore further, with lookahead and backtracking.

ToT consists of four components. For our response generation task, we implement them as follows:

1. **Thought decomposition:** the reasoning is split into three steps, and each level of the tree holds the thoughts for one step:
    * Step 1 (Intent & Constraints): which hotel or restaurant, and which aspect of it, is the user asking about?
    * Step 2 (Evidence Synthesis): do the reviews all agree, is there a single outlier, or are the opinions mixed?
    * Step 3 (Final Response): the response to the user, wrapped in `<answer></answer>` tags.
2. **Thought generation:** for each step, Qwen samples `k_candidates` different thoughts, given the dialogue, the reviews, and the thoughts so far.
3. **State evaluation:** Qwen judges how promising each (partial) path is. We implement the two strategies from the lecture (valuing each state independently, and voting across states) and a variant of the first that uses the model's token probabilities.
4. **Search algorithm:** breadth-first search (BFS) keeps the `beam_width` best paths at every step, while depth-first search (DFS) follows the most promising path first, prunes weak branches, and backtracks when a branch fails.

Note that, unlike the zero-shot, few-shot, and chain-of-thought prompts above, ToT also gets the **reviews** (the knowledge from the DSTC11 data, as in assignment 1) as evidence to reason over. Keep this in mind when you compare its scores to the other techniques.

**NOTE:** ToT is expensive. Every node in the tree requires model calls: with the default settings, BFS makes around 30 model calls per dialogue, compared to a single call for zero-shot prompting. The code prints the number of model calls for every dialogue. We therefore run ToT on only 5 dialogues. Make sure that your GPU accelerator is turned on.

<span style="background:yellow">__Q12:__ Review the Tree-of-Thoughts (ToT) code below. The tasks are marked with #TODO, but the full implementation is already provided to help you understand how ToT works. You do not need to write or complete any code. Just review the tasks and their implementations.</span>

The tasks covered in the code are:

- In **TOT_STEPS**, write the instructions for the three reasoning steps described above. The instruction for step 3 must ask the model to wrap its response in `<answer></answer>` tags.
- Complete **rank_candidates**, which sorts the candidate paths by their score. Both search algorithms use this function.
- In **run_tot_bfs**, keep the `beam_width` best paths.
- In **vote_states**, count the votes of the model. Keep in mind that the candidates are shuffled before every vote.
- Run ToT on the same 5 dialogues that you used for the manual evaluation in Q6: with BFS and the "logit" evaluator, with BFS and the "vote" evaluator, and with DFS.

### Knowledge

ToT reasons over the reviews, so we first load the knowledge base. As in the previous assignments, we only use the reviews (not the FAQs).

* **knowledge2sources:** looks up every review sentence referenced in the `knowledge` labels of a dialogue in `knowledge.json`, and returns these sentences as a list of strings. FAQ entries are skipped.

In [ ]:
import random
import time
import torch

with open('knowledge.json', 'r') as f:
    knowledge_base=json.load(f)


def knowledge2sources(knowledge):
    """
    Transform the knowledge labels of a dialogue into a list of review sentences called `sources`. Returns this list.
    """
    sources=[]
    for k in knowledge:
        ## We only use reviews: FAQ entries are skipped
        if k.get("doc_type", "review") != "review":
            continue
        try:
            domain = str(k["domain"])
            entity_id = str(k["entity_id"])
            doc_id = str(k["doc_id"])
            sent_id = str(k.get("sent_id", k.get("sentence_id")))
            source = knowledge_base[domain][entity_id]["reviews"][doc_id]["sentences"][sent_id]
            sources.append(source)
        except (KeyError, TypeError):
            continue
    return sources

Let's check the function on the first dialogue that has knowledge labels:

In [ ]:
sample_label = next(l for l in labels if l.get('knowledge'))
print(sample_label['knowledge'])

knowledge2sources(sample_label['knowledge'])

### LLM functions for ToT

`query_qwen` always generates with the model's default generation settings and up to 2000 new tokens. ToT needs more control: thoughts must be *sampled*, otherwise all candidate thoughts for a step would be identical, while evaluations should be *deterministic* (greedy decoding). One of the evaluators also needs direct access to the model to read its token probabilities.

* **prepare_model_input:** applies the chat template of the selected model (Qwen3-1.7B or Qwen2.5-3B, exactly as in `query_qwen`) and tokenizes the messages. Returns the model, the tokenizer, and the model inputs. It also counts every model call in **n_model_calls**, so that we can measure the cost of ToT.
* **query_qwen_tot:** the same as `query_qwen`, but with a configurable `max_new_tokens` and `temperature`. With `temperature=0`, it uses greedy decoding; with `temperature>0`, it samples (with top-p and top-k), so repeated calls give different outputs.

In [ ]:
n_model_calls = 0  # counts every model call (generation or forward pass), to measure the cost of ToT


def prepare_model_input(messages, model_name):
    """
    Apply the chat template of the selected model to the messages and tokenize them (as in `query_qwen`).
    Returns the model, the tokenizer and the model inputs.
    """
    global n_model_calls
    n_model_calls += 1

    if model_name == "Qwen3-1.7B":
        model, tokenizer = get_qwen3_model()

        text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
        )

    elif model_name == "Qwen2.5-3B":
        model, tokenizer = get_qwen2_model()

        text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
        )

    else:
        raise ValueError(f"Unknown model_name '{model_name}'. Use 'Qwen3-1.7B' or 'Qwen2.5-3B'.")

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    return model, tokenizer, model_inputs


def query_qwen_tot(messages, model_name, max_new_tokens=300, temperature=0.0):
    """
    Same as `query_qwen`, but with a configurable number of new tokens and temperature.
    temperature=0 gives greedy (deterministic) decoding; temperature>0 samples.
    Returns the decoded output.
    """
    model, tokenizer, model_inputs = prepare_model_input(messages, model_name)

    if temperature > 0:
        sampling_args = {"do_sample": True, "temperature": temperature, "top_p": 0.9, "top_k": 20}
    else:
        ## Switch off the sampling defaults from the model's generation config
        sampling_args = {"do_sample": False, "temperature": None, "top_p": None, "top_k": None}

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            **sampling_args
        )

    output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]
    output = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    return output

### Thought generation

* **TOT_STEPS:** the instructions for the three reasoning steps. A node at depth *d* of the tree holds one thought for step *d*.
* **format_reviews**, **format_history**, **format_path:** turn the reviews, the dialogue history, and a (partial) reasoning path into text for the prompts. **shorten** puts a thought on a single line and cuts it off, to keep the printed search trace readable.
* **generate_thoughts:** samples `k_candidates` thoughts for the next step, given the dialogue, the reviews, and the path so far. Duplicate samples are discarded and sampled again (up to twice as many attempts as needed), so that sibling nodes actually differ.

In [ ]:
## The three reasoning steps: a node at depth d of the tree holds one thought for step d
#TODO: write the instruction for each of the three steps (replace "##here##").
#      The instruction for step 3 must ask the model to wrap its response in <answer></answer> tags.
#      Tip: limit the length of each thought (e.g., "Keep it under 80 words"), as longer thoughts make the search slower.
TOT_STEPS = [
    "Step 1 (Intent & Constraints): Identify the exact hotel/restaurant entity, the subjective aspect the user is asking about, and any user constraints from the conversation. Keep it under 80 words.",
    "Step 2 (Evidence Synthesis): Analyze the provided reviews for that aspect. State whether the reviews all agree, have a single outlier, or are mixed, and summarize the facts. Keep it under 100 words.",
    "Step 3 (Final Response): Write the final concise, natural conversational response (at most 3 sentences) to the user, based strictly on Step 1 and Step 2. Output ONLY the response wrapped in <answer>...</answer> tags, with no text outside the tags."
]

THOUGHT_MAX_NEW_TOKENS = 320  # enough room for a full step-2 analysis or <answer> block
THOUGHT_TEMPERATURE = 0.7     # sampling makes the candidate thoughts differ


def format_reviews(sources):
    return "\n".join(f"- {s}" for s in sources) if sources else "No specific reviews available."

def format_history(history):
    return "\n".join(f"{'User' if ut['speaker'] == 'U' else 'Assistant'}: {ut['text']}" for ut in history)

def format_path(path):
    if not path:
        return "None yet."
    return "\n".join(f"Step {i + 1} thought: {thought}" for i, thought in enumerate(path))

def shorten(text, max_chars=200):
    text = " ".join(text.split())
    return text if len(text) <= max_chars else text[:max_chars] + "..."


def generate_thoughts(history, sources, path, step_idx, model_name, k_candidates=3):
    """
    Sample `k_candidates` different thoughts for step `step_idx`, continuing the (partial) reasoning `path`.
    Returns the list of thoughts.
    """
    messages = [
        {"role": "system", "content": (
            "You are generating reasoning steps in a Tree-of-Thoughts search for a dialogue system "
            "about hotels and restaurants.\n"
            f"Reviews:\n{format_reviews(sources)}"
        )},
        {"role": "user", "content": (
            f"Dialogue History:\n{format_history(history)}\n\n"
            f"Previous Thoughts:\n{format_path(path)}\n\n"
            f"Current Task:\n{TOT_STEPS[step_idx]}\n\n"
            "Respond with ONLY the output for this current step."
        )}
    ]

    thoughts = []
    seen = set()
    for attempt in range(2 * k_candidates):
        if len(thoughts) == k_candidates:
            break
        thought = query_qwen_tot(messages, model_name, max_new_tokens=THOUGHT_MAX_NEW_TOKENS, temperature=THOUGHT_TEMPERATURE)

        ## Compare thoughts while ignoring differences in case, punctuation and spacing
        key = "".join(ch if ch.isalnum() else " " for ch in thought.lower())
        key = " ".join(key.split())
        if key and key not in seen:  # skip empty and duplicate thoughts
            seen.add(key)
            thoughts.append(thought)

    if not thoughts:  # never return an empty list to the search
        thoughts.append(query_qwen_tot(messages, model_name, max_new_tokens=THOUGHT_MAX_NEW_TOKENS, temperature=0))
    return thoughts

### State evaluation

The state evaluator is the heuristic that guides the search: it decides which paths are worth exploring further. In the lecture, we saw two strategies, *value* (judge each state independently) and *vote* (compare the states with each other). We implement both, plus a variant of the first:

* **evaluate_state_logit** (`eval_method="logit"`, the default): Qwen is asked to rate a path from 1 to 5, but instead of letting it generate an answer, we read the probabilities of the tokens "1" to "5" at the first output position and compute the *expected* rating, rescaled to [0, 1]. This takes a single forward pass, is deterministic, and gives continuous scores, so candidates (almost) never tie.
* **evaluate_state_independent** (`eval_method="independent"`): Qwen judges a single path and returns a JSON object with a verdict (*sure / likely / impossible*) and a confidence score, which are combined into one score in [0, 1]. **parse_independent_eval** extracts these from the output and returns `None` if it cannot be parsed, in which case the state gets a neutral score of 0.5.
* **vote_states** (`eval_method="vote"`): Qwen sees all candidate paths at once and picks the best one. This is repeated `n_votes` times, with the candidates shuffled every time to cancel position bias. A candidate's score is its share of the votes. **extract_number** finds the number of the state the model voted for.

Two more helpers:

* **score_candidates:** scores a list of candidate paths with the evaluator selected by `eval_method`.
* **log_if_flat:** adds a warning to the search trace when all candidates get the same score. In that case, the evaluator did not discriminate between the candidates, and the "best" one is simply the first sample. Generated verdicts and votes tie much more often than the probability-based scores.

In [ ]:
## Evaluator 1 ("logit"): the expected 1-5 rating, read from the probabilities of the rating tokens
RATING_DIGITS = ["1", "2", "3", "4", "5"]

def evaluate_state_logit(history, sources, path, model_name):
    """
    Ask Qwen to rate the reasoning `path` from 1 to 5, and read the probabilities of the tokens "1"-"5"
    at the first output position (one forward pass, no generation).
    Returns the expected rating, rescaled to [0, 1].
    """
    messages = [
        {"role": "system", "content": (
            "You are a strict evaluator in a Tree-of-Thoughts search for a review-grounded dialogue system. "
            "Rate how likely the current reasoning is to lead to a final response that is (a) faithful to the "
            "reviews and (b) directly answers the user's last request.\n"
            "Use the full scale: 1 = unfaithful or off-topic, 2 = mostly wrong, 3 = partially correct, "
            "4 = good, 5 = fully faithful and on track.\n"
            "Reply with a single digit from 1 to 5 and nothing else."
        )},
        {"role": "user", "content": (
            f"User Query: {history[-1]['text']}\n\n"
            f"Reviews:\n{format_reviews(sources)}\n\n"
            f"Current Reasoning State:\n{format_path(path)}\n\n"
            "Rating (1-5):"
        )}
    ]
    model, tokenizer, model_inputs = prepare_model_input(messages, model_name)

    digit_ids = []
    for digit in RATING_DIGITS:
        token_ids = tokenizer.encode(digit, add_special_tokens=False)
        assert len(token_ids) == 1, f"'{digit}' is not a single token for this tokenizer"
        digit_ids.append(token_ids[0])

    with torch.no_grad():
        logits = model(**model_inputs, use_cache=False).logits[0, -1, :]  # scores for the first output token

    probs = torch.softmax(logits[digit_ids].float(), dim=-1).tolist()  # P(rating), renormalized over "1"-"5"
    ratings = range(1, len(RATING_DIGITS) + 1)
    expected_rating = sum(p * rating for p, rating in zip(probs, ratings))
    return (expected_rating - 1) / (len(RATING_DIGITS) - 1)


## Evaluator 2 ("independent"): a sure/likely/impossible verdict plus a confidence score
VERDICT_SCORES = {"sure": 1.0, "likely": 0.5, "impossible": 0.1}

def parse_independent_eval(raw_eval):
    """
    Extract the verdict and the score from the evaluator's JSON output and combine them into one score in [0, 1].
    Returns None if nothing trustworthy could be parsed.
    """
    ## The JSON object runs from the first "{" to the next "}" (with or without <JSON> tags or code fences around it)
    data = None
    start = raw_eval.find("{")
    end = raw_eval.find("}", start)
    if start != -1 and end != -1:
        try:
            data = json.loads(raw_eval[start:end + 1])
        except json.JSONDecodeError:
            data = None

    if isinstance(data, dict):
        verdict_score = VERDICT_SCORES.get(str(data.get("verdict", "")).strip().lower())
        try:
            confidence = max(0.0, min(1.0, float(data.get("score"))))
        except (TypeError, ValueError):
            confidence = None
        if verdict_score is not None and confidence is not None:
            return 0.5 * verdict_score + 0.5 * confidence
        if verdict_score is not None:
            return verdict_score
        if confidence is not None:
            return confidence

    ## Fallback for broken JSON: only trust the word right after "verdict"
    text = raw_eval.lower()
    position = text.find("verdict")
    if position != -1:
        rest = text[position + len("verdict"):]
        for symbol in "\"':=,.{}":
            rest = rest.replace(symbol, " ")
        words = rest.split()
        if words and words[0] in VERDICT_SCORES:
            return VERDICT_SCORES[words[0]]
    return None


def evaluate_state_independent(history, sources, path, model_name):
    """
    Ask Qwen for a JSON verdict on the reasoning `path`.
    Returns a score in [0, 1] (0.5 if the output could not be parsed).
    """
    messages = [
        {"role": "system", "content": (
            "You are a State Evaluator in a Tree-of-Thoughts search. "
            "Judge whether the current reasoning steps are factually faithful to the reviews, "
            "accurately address the user's query, and are on track to produce a high-quality response.\n"
            "Return ONLY a JSON object with two fields:\n"
            '  "verdict": one of "sure", "likely", "impossible"\n'
            '  "score": a number between 0 and 1 reflecting your honest confidence\n'
            "Be critical and use the full range of scores."
        )},
        {"role": "user", "content": (
            f"User Query: {history[-1]['text']}\n\n"
            f"Reviews:\n{format_reviews(sources)}\n\n"
            f"Current Reasoning State:\n{format_path(path)}"
        )}
    ]
    raw_eval = query_qwen_tot(messages, model_name, max_new_tokens=120, temperature=0)
    score = parse_independent_eval(raw_eval)
    return 0.5 if score is None else score

In [ ]:
## Evaluator 3 ("vote"): Qwen picks the best of all candidate states
def extract_number(text):
    """
    Returns the number between the <best> and </best> tags, or else the first number in the text (None if there is none).
    """
    start = text.find("<best>")
    end = text.find("</best>", start)
    if start != -1 and end != -1:
        inside = text[start + len("<best>"):end].strip()
        if inside.isdecimal():
            return int(inside)

    for word in text.replace("<", " ").replace(">", " ").split():
        word = word.strip(".,:;!?()[]*\"'")
        if word.isdecimal():
            return int(word)
    return None


def vote_states(history, sources, candidate_paths, model_name, n_votes=3):
    """
    Let Qwen vote `n_votes` times for the best candidate path, shuffling the candidates before every vote
    to cancel position bias. Returns each candidate's share of the votes.
    """
    n = len(candidate_paths)
    if n == 1:
        return [1.0]

    votes = [0] * n  # votes[i] is the number of votes for candidate_paths[i]
    valid_votes = 0
    for _ in range(n_votes):
        ## Shuffle: "State 1" in the prompt is candidate_paths[order[0]], "State 2" is candidate_paths[order[1]], etc.
        order = list(range(n))
        random.shuffle(order)
        options = "\n\n".join(f"State {pos + 1}:\n{format_path(candidate_paths[i])}" for pos, i in enumerate(order))
        messages = [
            {"role": "system", "content": (
                "Compare the following candidate reasoning states given the user query and reviews. "
                "Select the state number that is most accurate and faithful to the reviews. "
                "Reply with ONLY the integer number of the best state inside <best>X</best> tags."
            )},
            {"role": "user", "content": (
                f"User Query: {history[-1]['text']}\n\n"
                f"Reviews:\n{format_reviews(sources)}\n\n"
                f"Candidate States:\n{options}"
            )}
        ]
        raw_vote = query_qwen_tot(messages, model_name, max_new_tokens=20, temperature=0)
        vote = extract_number(raw_vote)  # the number of the state the model picked (None if there is no number)

        #TODO: if `vote` is a valid state number (from 1 to n), add one vote for the candidate the model picked,
        #      and count it as a valid vote. Remember the shuffle: "State 1" in the prompt is candidate_paths[order[0]].
        if vote is not None and 1 <= vote <= n:
            votes[order[vote - 1]] += 1
            valid_votes += 1

    if valid_votes == 0:  # no usable vote: treat all candidates the same
        return [0.5] * n
    return [v / valid_votes for v in votes]


EVAL_METHODS = ["logit", "vote", "independent"]

def score_candidates(history, sources, candidate_paths, model_name, eval_method="logit"):
    """
    Score a list of candidate paths with the selected evaluator.
    Returns one score in [0, 1] per path (higher is better).
    """
    if eval_method == "logit":
        return [evaluate_state_logit(history, sources, path, model_name) for path in candidate_paths]
    if eval_method == "independent":
        return [evaluate_state_independent(history, sources, path, model_name) for path in candidate_paths]
    if eval_method == "vote":
        return vote_states(history, sources, candidate_paths, model_name)
    raise ValueError(f"Unknown eval_method '{eval_method}'. Choose from {EVAL_METHODS}.")


def log_if_flat(scores, label, tree_log):
    """
    Add a warning to the search trace when all candidates have the same score.
    """
    if len(scores) > 1 and max(scores) - min(scores) < 1e-6:
        tree_log.append(f"  !! {label}: all {len(scores)} candidates scored {scores[0]:.2f}, "
                        "so the evaluator did not discriminate and the first sample wins")

### Extracting the final answer

* **clean_answer:** removes leftover tags, and labels such as "Step 3 (Final Response):", from a response.
* **extract_tot_answer:** returns the response between the `<answer>` and `</answer>` tags of a step-3 thought, or an empty string if there is no complete answer block.
* **finalize_answer:** a fallback for when the best step-3 thought has no complete `<answer>` block (e.g., because the generation was cut off or ignored the format). Qwen writes the response once more with greedy decoding, using the reviews and the chosen reasoning path as guidance.
* **get_final_answer:** returns the final answer of the chosen path, using the fallback when needed. **has_text** checks that an answer is not empty.

In [ ]:
ANSWER_LABELS = ["step 3", "final answer", "final response"]

def clean_answer(text):
    """
    Remove leftover tags, and a label such as "Step 3 (Final Response):" at the start of a response.
    Returns the cleaned response.
    """
    if "</think>" in text:  # keep only what comes after Qwen3's thinking part, if there is one
        text = text.split("</think>")[-1]
    for tag in ["<answer>", "</answer>", "<output>", "</output>", "<reasoning>", "</reasoning>", "<think>", "</think>"]:
        text = text.replace(tag, "")
    text = text.strip()

    ## Remove a label like "Step 3 (Final Response):" or "**Final Answer:**" (but keep the bold in "Final Answer: **Yes**")
    colon = text.find(":")
    if colon != -1:
        label = text[:colon].strip("#*_ \n").lower()
        if label in ANSWER_LABELS or (label.startswith("step 3") and label.endswith(")")):
            label_is_bold = text.lstrip("# ").startswith(("**", "__"))
            text = text[colon + 1:].strip()
            if label_is_bold:  # also remove the closing ** of the bold label
                text = text.lstrip("*_").strip()
    return text


def extract_tot_answer(thought):
    """
    Returns the response between the <answer> and </answer> tags of a step-3 thought, or "" if there is no complete answer block.
    """
    start = thought.find("<answer>")
    end = thought.find("</answer>", start)
    if start == -1 or end == -1:
        return ""
    return clean_answer(thought[start + len("<answer>"):end])


def finalize_answer(history, sources, path, model_name):
    """
    Fallback: generate the response with greedy decoding, using the reviews and the reasoning `path` as guidance.
    """
    instruction = (
        "You are a helpful assistant for hotels and restaurants. "
        "Answer the user's last request concisely, grounded ONLY in these reviews:\n"
        f"{format_reviews(sources)}\n\n"
        "Use this prior analysis as guidance:\n"
        f"{format_path(path)}\n\n"
        "Output ONLY the final response to the user, with no tags, labels or meta-commentary."
    )
    messages=[]
    messages.append({"role": "system", "content": instruction})
    for ut in history:
        role = "user" if ut['speaker']=="U" else "assistant"
        messages.append({"role": role, "content": ut['text']})

    return clean_answer(query_qwen_tot(messages, model_name, max_new_tokens=200, temperature=0))


def has_text(answer):
    """
    Returns True if the answer contains at least one letter or digit.
    """
    return any(ch.isalnum() for ch in answer)


def get_final_answer(history, sources, path, model_name):
    """
    Returns the final answer of the reasoning `path`.
    """
    answer = extract_tot_answer(path[-1])
    if not has_text(answer):  # no usable <answer> block
        answer = finalize_answer(history, sources, path, model_name)
    if not has_text(answer):  # very unlikely, but an empty answer would break the ROUGE computation
        answer = "I'm sorry, I don't have enough information to answer that."
    return answer

### Search algorithm 1: Breadth-first search (BFS)

* **rank_candidates:** pairs every candidate path with its score, and sorts the pairs from the highest to the lowest score. Both search algorithms use this function.
* **run_tot_bfs:** explores the tree level by level. At every step, each path in the beam is extended with `k_candidates` new thoughts, all new paths are scored, and only the `beam_width` best paths are kept. After the last step, the final answer is taken from the best path. Returns the search trace and the final answer.
* **format_trace:** combines the search log with the full thoughts of the chosen path into a printable trace.

With `k_candidates=3` and `beam_width=2`, the tree has 3 nodes at depth 1 and 6 nodes at depths 2 and 3.

In [ ]:
def rank_candidates(candidates, scores):
    """
    Pair every candidate path with its score.
    Returns the (path, score) pairs, sorted from the highest to the lowest score.
    """

    #TODO: complete this function. Hint: zip() pairs up two lists, and sorted() has the arguments `key` and `reverse`.
    ranked = sorted(zip(candidates, scores), key=lambda pair: pair[1], reverse=True)
    return ranked


def format_trace(tree_log, best_path):
    return "\n".join(["TREE SEARCH:"] + tree_log + ["", "BEST PATH:", format_path(best_path)])


def run_tot_bfs(history, knowledge, model_name, k_candidates=3, beam_width=2, eval_method="logit"):
    """
    Tree-of-Thoughts with breadth-first search. Returns the search trace and the final answer.
    """
    sources = knowledge2sources(knowledge)
    tree_log = []
    beam = [[]]  # the paths we keep; the search starts from the empty path (the root of the tree)

    for step_idx in range(len(TOT_STEPS)):
        ## 1. Expand: extend every path in the beam with k_candidates new thoughts
        candidates = []
        for path in beam:
            for thought in generate_thoughts(history, sources, path, step_idx, model_name, k_candidates):
                candidates.append(path + [thought])

        ## 2. Evaluate all new paths
        scores = score_candidates(history, sources, candidates, model_name, eval_method)

        ## 3. Keep the beam_width best paths
        ranked = rank_candidates(candidates, scores)
        for rank, (path, score) in enumerate(ranked):
            status = "KEPT" if rank < beam_width else "dropped"
            tree_log.append(f"[BFS depth {step_idx + 1} | cand {rank + 1} | score {score:.2f} | {status}] {shorten(path[-1])}")
        log_if_flat(scores, f"BFS depth {step_idx + 1}", tree_log)

        #TODO: keep only the beam_width best paths (without their scores) in `beam`
        beam = [path for path, score in ranked[:beam_width]]

    best_path = beam[0]
    final_answer = get_final_answer(history, sources, best_path, model_name)
    return format_trace(tree_log, best_path), final_answer

### Search algorithm 2: Depth-first search (DFS)

* **run_tot_dfs:** always expands the most promising thought first and goes deeper until it reaches step 3. Thoughts that score below `prune_threshold` are pruned. If all children of a node are pruned (or their subtrees fail), the search **backtracks** to the next-best sibling of that node. Expanding a node costs `k_candidates` generations and evaluations, so `max_expansions` limits the number of expanded nodes. Returns the search trace and the final answer.
* **greedy_complete:** if the budget runs out before a complete path is found, the deepest, best-scoring partial path is completed by taking the best new thought at every remaining step.

The scale of the scores depends on the evaluator, so each evaluator has its own default threshold (`DEFAULT_PRUNE_THRESHOLDS`): 0.5 for "logit" (an expected rating of 3 out of 5) and for "independent", and 0.3 for "vote" (a candidate needs at least one of the three votes). If you change the evaluator or the threshold, check the scores in the trace.

In [ ]:
## The scale of the scores differs per evaluator, so each has its own default pruning threshold
DEFAULT_PRUNE_THRESHOLDS = {"logit": 0.5, "independent": 0.5, "vote": 0.3}


def greedy_complete(history, sources, path, model_name, k_candidates, eval_method, tree_log):
    """
    Complete a partial path by taking the best-scoring new thought at every remaining step. Returns the complete path.
    """
    path = list(path)
    for step_idx in range(len(path), len(TOT_STEPS)):
        candidates = [path + [t] for t in generate_thoughts(history, sources, path, step_idx, model_name, k_candidates)]
        scores = score_candidates(history, sources, candidates, model_name, eval_method)
        path, score = rank_candidates(candidates, scores)[0]
        tree_log.append(f"[Greedy depth {step_idx + 1} | score {score:.2f}] {shorten(path[-1])}")
    return path


def run_tot_dfs(history, knowledge, model_name, k_candidates=3, prune_threshold=None, eval_method="logit", max_expansions=10):
    """
    Tree-of-Thoughts with depth-first search, pruning and backtracking. Returns the search trace and the final answer.
    """
    if prune_threshold is None:
        prune_threshold = DEFAULT_PRUNE_THRESHOLDS[eval_method]
    sources = knowledge2sources(knowledge)
    n_steps = len(TOT_STEPS)
    tree_log = []
    search = {"complete_path": None, "best_partial": ([], -1.0), "expansions": 0}

    def dfs(path, step_idx):
        """
        Explore the subtree below `path`. Returns True as soon as a complete path is found, False if the subtree fails.
        """
        if step_idx == n_steps:  # all steps are done: we found a complete path
            search["complete_path"] = path
            return True
        if search["expansions"] >= max_expansions:  # the budget is used up
            return False
        search["expansions"] += 1

        indent = "  " * step_idx
        candidates = [path + [t] for t in generate_thoughts(history, sources, path, step_idx, model_name, k_candidates)]
        scores = score_candidates(history, sources, candidates, model_name, eval_method)
        log_if_flat(scores, f"DFS depth {step_idx + 1}", tree_log)

        ## Visit the children from the most to the least promising one
        for child, score in rank_candidates(candidates, scores):
            tree_log.append(f"{indent}[DFS depth {step_idx + 1} | score {score:.2f}] {shorten(child[-1])}")

            ## Remember the deepest, best-scoring path, in case no complete path is found
            best_path, best_score = search["best_partial"]
            if (len(child), score) > (len(best_path), best_score):
                search["best_partial"] = (child, score)

            if score < prune_threshold:
                tree_log.append(f"{indent}  -> pruned (score {score:.2f} < {prune_threshold})")
                continue
            if dfs(child, step_idx + 1):
                return True
            if search["expansions"] < max_expansions:
                tree_log.append(f"{indent}  <- subtree failed, backtracking")
        return False

    dfs([], 0)

    if search["complete_path"] is not None:
        best_path = search["complete_path"]
    else:
        tree_log.append(f"== No complete path within {max_expansions} expansions: "
                        "completing the best partial path greedily ==")
        best_path = greedy_complete(history, sources, search["best_partial"][0], model_name,
                                    k_candidates, eval_method, tree_log)

    final_answer = get_final_answer(history, sources, best_path, model_name)
    return format_trace(tree_log, best_path), final_answer

### ToT model run and evaluation

* **run_pipeline_tot:** similar to *run_pipeline*, this function iterates over the dialogues until reaching *n_dialogues*. For each dialogue, it runs the selected search (`search="bfs"` or `search="dfs"`) with the selected evaluator, prints the search trace and the cost (the number of model calls and the time), and evaluates the final answer with ROUGE and BLEU. Finally, it reports the average scores with *final_evaluation*, together with the average number of model calls per dialogue. The random seed is fixed, so that the sampled thoughts are reproducible.

In the trace, each line shows the depth of a node, its score, what happened to it (kept, dropped, or pruned), and the start of its thought. The **BEST PATH** shows the full thoughts of the path that produced the final answer.

In [ ]:
def run_pipeline_tot(n_dialogues, model_name, search="bfs", eval_method="logit",
                     k_candidates=3, beam_width=2, prune_threshold=None, max_expansions=10, seed=3407):
    global n_model_calls

    if search not in ["bfs", "dfs"]:
        raise ValueError(f"Unknown search '{search}'. Use 'bfs' or 'dfs'.")
    if eval_method not in EVAL_METHODS:
        raise ValueError(f"Unknown eval_method '{eval_method}'. Choose from {EVAL_METHODS}.")

    ## Make the sampled thoughts and the shuffled votes reproducible
    random.seed(seed)
    torch.manual_seed(seed)

    all_bleus=[]
    all_rouges=[]
    all_predictions=[]
    all_references=[]
    all_model_calls=[]

    counter_dialogues = 0
    idx_dialogue = 0

    while counter_dialogues < n_dialogues and idx_dialogue < len(contexts):
        ## Get a valid dialogue
        chat = contexts[idx_dialogue]
        if 'response' not in labels[idx_dialogue]:
            idx_dialogue += 1
            continue
        gt_response = labels[idx_dialogue]['response']
        knowledge = labels[idx_dialogue].get('knowledge', [])
        counter_dialogues += 1
        idx_dialogue += 1

        print("Processing dialogue", counter_dialogues)
        print("USER:", chat[-1]['text'])
        if not knowledge2sources(knowledge):
            print("NOTE: no review sentences found for this dialogue, so ToT has no reviews to reason over.")

        ## Run the tree search, and count the model calls it makes
        n_model_calls = 0
        start_time = time.time()
        if search == "bfs":
            tree_trace, model_response = run_tot_bfs(chat, knowledge, model_name, k_candidates=k_candidates,
                                                     beam_width=beam_width, eval_method=eval_method)
        else:
            tree_trace, model_response = run_tot_dfs(chat, knowledge, model_name, k_candidates=k_candidates,
                                                     prune_threshold=prune_threshold, eval_method=eval_method,
                                                     max_expansions=max_expansions)
        print(tree_trace)
        print(f"COST: {n_model_calls} model calls, {time.time() - start_time:.0f} seconds")
        print()

        ## Evaluation
        rouge, bleu = evaluate_answers(model_response, gt_response)

        ## save the results
        all_predictions.append(model_response)
        all_references.append(gt_response)
        all_bleus.append(bleu)
        all_rouges.append(rouge)
        all_model_calls.append(n_model_calls)

    final_evaluation(all_bleus, all_rouges, all_predictions, all_references)
    print("Mean model calls per dialogue", np.mean(all_model_calls))

Let's run ToT with breadth-first search and the "logit" evaluator:

In [ ]:
n_dialogues = 5
model_name = "Qwen3-1.7B" ## or "Qwen2.5-3B": use the model you selected in Q5

run_pipeline_tot(n_dialogues, model_name, search="bfs", eval_method="logit")

Now with breadth-first search and the "vote" evaluator:

In [ ]:
run_pipeline_tot(n_dialogues, model_name, search="bfs", eval_method="vote")

And with depth-first search. Look for the pruned branches and backtracking steps in the trace:

In [ ]:
run_pipeline_tot(n_dialogues, model_name, search="dfs", eval_method="logit")

<span style="background:yellow">__Q13:__ Compare ToT (BFS and DFS) with zero-shot chain-of-thought (Q4) on these 5 dialogues, using the automatic metrics, your three manual criteria from Q6, and the cost of each method (the number of model calls per dialogue).</span>

Afterward, analyze the results:

- Do the evaluator's scores in the search traces agree with your own judgment of which thoughts are better? Give an example.
- Do the "logit" and "vote" evaluators prefer the same paths?
- Is the improvement (if any) worth the extra computation for this task? Keep in mind that ToT also gets the reviews, while the other prompts in this notebook do not.

```
# Your answer here
```